In [ ]:
import json
import re
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
import os

os.chdir(r"C:\Users\Hp\Downloads\Project 2026 DS")
BASE=Path(r"C:\Users\Hp\Downloads\Project 2026 DS")

LONDON_BOROUGHS=[
    "Barking and Dagenham","Barnet","Bexley","Brent","Bromley","Camden","City of London","Croydon","Ealing","Enfield","Greenwich","Hackney","Hammersmith and Fulham","Haringey","Harrow","Havering","Hillingdon",
    "Hounslow","Islington","Kensington and Chelsea","Kingston upon Thames","Lambeth","Lewisham","Merton","Newham","Redbridge","Richmond upon Thames","Southwark","Sutton","Tower Hamlets",
    "Waltham Forest","Wandsworth","Westminster"
]

print("BASE)

In [ ]:
all_las=gpd.read_file(BASE / "boundaryfile.geojson")
boroughs=all_las[all_las["LAD24NM"].isin(LONDON_BOROUGHS)].copy()

print(f"Found {len(boroughs)} / 33 London boroughs")
missing_boroughs=set(LONDON_BOROUGHS) - set(boroughs["LAD24NM"])
if missing_boroughs:
    print(" not matched in boundary file:",missing_boroughs)

In [ ]:
ACTIVITY_KEYWORD_MAP=[
    (["swim","aqua","diving","kayak","paddleboard","watersport"],"Swimming"),
    (["yoga","pilates","tai chi","meditation","barre"],"Yoga, Pilates & Studio"),
    (["badminton","squash","tennis","table tennis","pickleball","fives"],"Racquet Sports"),
    (["football","netball","basketball","cricket","rugby","volleyball","rounders","boccia","multisport","a-side","pitch","softball","baseball"],"Team Sports"),
    (["boxing","martial","karate","judo","boxercise","boxfit"],"Martial Arts"),
    (["bootcamp","outdoor","climb","bouldering","trampolin"],"Bootcamp & Outdoor Fitness"),
    (["run","athletic","walking","track"],"Walking / Running"),
    (["cycl","spin","rpm"],"Cycling"),
    (["personal training","1-2-1","pt "],"Personal Training"),
    (["dance","salsa","zumba","latin","ballroom","line danc"],"Dance"),
    (["gym","fitness","hiit","circuit","conditioning","strength","bodypump","bodycombat","bodyattack","bodybalance","bodystep","core","abs","legs bums","stretch","cx worx","grit","kettlebell","weights","toning","lifting","calisthenics","kickstart","hyrox","battle blast","pull up","functional","double impact","induction"],"Gym / Fitness"),
    (["class","aerobic","exercise","workout"],"Group Exercise"),
    (["ice skat"],"Ice Skating & Wheeled Sports"),
    (["sauna","spa","soft play","good boost land","sound bath","focus","activate"],"Wellness & Facility Access"),
]

def _normalize(text):
    return re.sub(r'[^a-z]','',str(text).lower())

def map_activity_type(activity_raw):
    if not activity_raw:
        return "Other / Unspecified"
    text=_normalize(activity_raw)
    for keywords,category in ACTIVITY_KEYWORD_MAP:
        if any(_normalize(kw) in text for kw in keywords if _normalize(kw)):
            return category
    return "Other / Unspecified"

In [ ]:
def load_json(path):
    with open(path,encoding="utf-8") as f:
        return json.load(f)

def _first_price(offers):
    if offers is None:
        return None,None
    if isinstance(offers,dict):
        offers=[offers]
    if not isinstance(offers,list) or len(offers) == 0:
        return None,None
    prices=[o.get("price") for o in offers if isinstance(o,dict) and o.get("price") is not None]
    if not prices:
        return None,None
    price=min(prices)
    return price,(price == 0)

def _get_activity_raw(d):
    activity_list=d.get("activity")
    if not activity_list:
        super_event=d.get("superEvent")
        if isinstance(super_event,dict):
            activity_list=super_event.get("activity")
    if not activity_list:
        instance_of_course=d.get("instanceOfCourse")
        if isinstance(instance_of_course,dict):
            activity_list=instance_of_course.get("activity")
    if not activity_list:
        category=d.get("category")
        if category:
            if not isinstance(category,list):
                category=[category]
            first_cat=category[0]
            if isinstance(first_cat,str):
                return first_cat
    if not activity_list:
        return None
    if not isinstance(activity_list,list):
        activity_list=[activity_list]
    first=activity_list[0]
    return first.get("prefLabel") if isinstance(first,dict) else None

def _extract_times_and_days(start_date_str,end_date_str):
    if not start_date_str:
        return None,None,None
    try:
        start=pd.to_datetime(start_date_str)
        end=pd.to_datetime(end_date_str) if end_date_str else None
        start_time=start.strftime("%H:%M")
        end_time=end.strftime("%H:%M") if end is not None else None
        day_of_week=start.strftime("%A")
        return start_time,end_time,day_of_week
    except Exception:
        return None,None,None

def _get_source_operator(d):
    provider=d.get("provider") or d.get("organizer")
    if not provider:
        super_event=d.get("superEvent")
        if isinstance(super_event,dict):
            provider=super_event.get("organizer")
    if isinstance(provider,dict):
        return provider.get("name")
    return None

def _clean_id(item,d):
    raw_id=item.get("id") or d.get("identifier") or d.get("@id")
    if raw_id is None:
        return None
    raw_id=str(raw_id)
    if raw_id.startswith("http"):
        raw_id=raw_id.rstrip("/").split("/")[-1]
    return raw_id

def build_parent_lookup(parent_items,parent_id_field="@id"):
    lookup={}
    for item in parent_items:
        d=item.get("data",{})
        loc=d.get("location") or {}
        geo=loc.get("geo") or {}
        address=loc.get("address") or {}
        price,is_free=_first_price(d.get("offers") or d.get("offer"))

        lookup[d.get(parent_id_field)]={
            "location_name": loc.get("name") or d.get("name"),
            "lat": geo.get("latitude"),
            "lon": geo.get("longitude"),
            "postcode": address.get("postalCode"),
            "street_address": address.get("streetAddress"),
            "activity_raw": _get_activity_raw(d),
            "price_gbp": price,
            "is_free": is_free,
            "source_operator": _get_source_operator(d),
            "name": d.get("name"),
            "url": d.get("url"),
        }
    return lookup

def build_row(item,d,parent_lookup_entry,provider_group,session_id_prefix):
    p=parent_lookup_entry or {}
    clean_id=_clean_id(item,d)

    loc=d.get("location") or {}
    geo=loc.get("geo") or {}
    address=loc.get("address") or {}

    lat=geo.get("latitude",p.get("lat"))
    lon=geo.get("longitude",p.get("lon"))
    location_name=loc.get("name") or d.get("name") or p.get("location_name")
    postcode=(address.get("postalCode") or p.get("postcode") or "").upper().strip() or None
    street_address=address.get("streetAddress") or p.get("street_address")

    child_price,child_free=_first_price(d.get("offers") or d.get("offer"))
    price_gbp=child_price if child_price is not None else p.get("price_gbp")
    is_free=child_free if child_free is not None else p.get("is_free")

    activity_raw=_get_activity_raw(d) or p.get("activity_raw")
    source_operator=_get_source_operator(d) or p.get("source_operator")

    start_time,end_time,day_of_week=_extract_times_and_days(d.get("startDate"),d.get("endDate"))

    return {
        "session_id": f"{session_id_prefix}_{clean_id}",
        "provider_group": provider_group,
        "source_operator": source_operator,
        "name": d.get("name") or p.get("name"),
        "activity_raw": activity_raw,
        "is_free": is_free,
        "price_gbp": price_gbp,
        "location_name": location_name,
        "street_address": street_address,
        "postcode": postcode,
        "lat": lat,
        "lon": lon,
        "start_time": start_time,
        "end_time": end_time,
        "days_of_week": day_of_week,
        "session_count": 1,
        "is_online": lat is None and lon is None and location_name is None,
        "url": d.get("url") or p.get("url"),
    }

def join_children(child_items,parent_lookup,link_field,provider_group,session_id_prefix):
    rows=[]
    for item in child_items:
        d=item.get("data",{})
        parent_entry=parent_lookup.get(d.get(link_field),{})
        rows.append(build_row(item,d,parent_entry,provider_group,session_id_prefix))
    df=pd.DataFrame(rows)
    matched=df["lat"].notna().sum()
    print(f"  joined {len(df)} rows, {matched} matched to a venue ({matched/len(df)*100:.1f}%)")
    return df

def flatten_standalone(items,provider_group,session_id_prefix):
    rows=[]
    for item in items:
        d=item.get("data",{})
        rows.append(build_row(item,d,{},provider_group,session_id_prefix))
    df=pd.DataFrame(rows)
    matched=df["lat"].notna().sum()
    print(f"  {len(df)} rows, {matched} have coordinates ({matched/len(df)*100:.1f}%)")
    return df

print("Functions ready.")

In [ ]:
pl_sessions=load_json(BASE / "Places_Leisure" / "ScheduledSession.json")
pl_series=load_json(BASE / "Places_Leisure" / "SessionSeries.json")
pl_slots=load_json(BASE / "Places_Leisure" / "Slot.json")
pl_facilities=load_json(BASE / "Places_Leisure" / "FacilityUse.json")
pl_courses=load_json(BASE / "Places_Leisure" / "CourseInstance.json")

pl_series_lookup=build_parent_lookup(pl_series)
pl_facility_lookup=build_parent_lookup(pl_facilities)

print("Places leisure-session:")
df_pl_session=join_children(pl_sessions,pl_series_lookup,"superEvent","Places Leisure","PL")

print("Places leisure-slot:")
df_pl_slot=join_children(pl_slots,pl_facility_lookup,"facilityUse","Places Leisure","PL")

print("Places leisure-CourseInstance:")
df_pl_courses=flatten_standalone(pl_courses,"Places Leisure","PL")

In [ ]:
ea_sessions=load_json(BASE / "Everyone_Active" / "ScheduledSession.json")
ea_series=load_json(BASE / "Everyone_Active" / "SessionSeries.json")
ea_facilities=load_json(BASE / "Everyone_Active" / "FacilityUse.json")
ea_courses=load_json(BASE / "Everyone_Active" / "CourseInstance.json")

ea_series_lookup=build_parent_lookup(ea_series)

print("everyone Active-session:")
df_ea_session=join_children(ea_sessions,ea_series_lookup,"superEvent","Everyone Active","EA")

print("everyone Active-facilityUse:")
df_ea_facilities=flatten_standalone(ea_facilities,"Everyone Active","EA")

print("everyone Active-courseInstance:")
df_ea_courses=flatten_standalone(ea_courses,"Everyone Active","EA")

In [ ]:
def filter_to_london(df,boroughs_gdf):
    df=df.dropna(subset=["lat","lon"]).copy()
    gdf=gpd.GeoDataFrame(df,geometry=[Point(xy) for xy in zip(df.lon,df.lat)],crs="EPSG:4326")
    gdf=gdf.to_crs(boroughs_gdf.crs)
    joined=gpd.sjoin(gdf,boroughs_gdf[["LAD24NM","geometry"]],how="left",predicate="within")
    london_only=joined[joined["index_right"].notna()].copy()
    london_only=london_only.drop(columns=["geometry","index_right"]).rename(columns={"LAD24NM": "borough"})
    print(f"  {len(london_only)} / {len(df)} rows fall in borough")
    return london_only

all_frames={
    "Places Leisure / Session": df_pl_session,
    "Places Leisure / Slot": df_pl_slot,
    "Places Leisure / CourseInstance": df_pl_courses,
    "Everyone Active / Session": df_ea_session,
    "Everyone Active / FacilityUse": df_ea_facilities,
    "Everyone Active / CourseInstance": df_ea_courses,
}

london_frames=[]
for label,df in all_frames.items():
    print(f"{label}:")
    london_frames.append(filter_to_london(df,boroughs))

combined=pd.concat(london_frames,ignore_index=True)
combined["activity_type"]=combined["activity_raw"].apply(map_activity_type)

COLUMN_ORDER=[
    "session_id","provider_group","source_operator","name",
    "activity_type","activity_raw","is_free","price_gbp",
    "location_name","street_address","postcode","borough","lat","lon",
    "start_time","end_time","days_of_week","session_count",
    "is_online","url",
]
combined=combined[COLUMN_ORDER]

combined.to_csv(BASE / "Person2_Combined_London.csv",index=False)
print("Saved Person2_Combined_London.csv ",combined.shape)

In [ ]:
print("missing values")
print( combined.shape)
print((combined.isna().sum() / len(combined) * 100).round(1))

print("\n Activity type")
print(combined["activity_type"].value_counts())
print("\nOther %:",round((combined["activity_type"] == "Other / Unspecified").sum() / len(combined) * 100,1))

print("\n provider ")
print(combined.groupby(["provider_group","activity_type"]).size().unstack(fill_value=0))

print("\n borough ")
print(combined["borough"].value_counts())

In [ ]:
venue_activity_summary=(
    combined
    .groupby(
        ["location_name","name","start_time","end_time","days_of_week","activity_raw","activity_type","borough","provider_group","source_operator"],
        dropna=False)
    .agg(n_occurrences=("session_id","count"),
        lat=("lat","first"),
        lon=("lon","first"),
        postcode=("postcode","first"),
        street_address=("street_address","first"),
        is_free=("is_free","first"),
        price_gbp=("price_gbp","min"),
        url=("url","first"),
        first_session_id=("session_id","first"),)
    .reset_index()
    .sort_values(["borough","location_name","activity_raw"]))

venue_activity_summary.insert(0,"session_id","V_" + venue_activity_summary.index.astype(str).str.zfill(5))

print(" rows:",len(combined))
print("Grouped rows",len(venue_activity_summary))

venue_activity_summary.to_csv(BASE / "Person2_VenueActivity_London.csv",index=False)
print("Saved Person2_VenueActivity_London.csv")

In [ ]:
df=pd.read_csv(BASE / "Person2_VenueActivity_London.csv")
df=df.rename(columns={"lat": "latitude","lon": "longitude","n_occurrences": "session_count",})
df["is_online"]=False
df=df.drop(columns=["street_address","first_session_id"])
SCHEMA_COLUMNS=["session_id","provider_group","source_operator","name","activity_type","activity_raw","is_free","price_gbp","location_name","postcode","borough","latitude","longitude",
    "start_time","end_time","days_of_week","session_count","is_online","url"]
df=df[SCHEMA_COLUMNS]

df.to_csv(BASE / "Person2_VenueActivity_London.csv",index=False)
print("Saved",df.shape)
print(df.columns.tolist())